Notebooks 15 and 16 covered micro-batch execution, sources, sinks, windowed aggregations, and watermarking. All of those are forms of **stateful streaming** — Spark maintains intermediate state across micro-batches so that results can span more than one batch.

This notebook goes beyond built-in `groupBy + agg` to explore the full spectrum of stateful patterns:

```
Stateless           Structured                  Arbitrary
filter/select  ──►  groupBy + window  ──►  mapGroupsWithState
                    (Spark manages state)       flatMapGroupsWithState
                                               (you manage state)
```

In this notebook you will learn:
- **Running aggregations** — cumulative totals that grow with every micro-batch
- **Session windows** — variable-duration windows that close on inactivity gaps
- **`mapGroupsWithState`** — arbitrary per-key state with a user-defined update function
- **`flatMapGroupsWithState`** — same, but can emit zero or many rows per key per batch
- **Timeouts** — how to expire state for keys that stop receiving data
- **State store backends** — HDFS-backed (default) vs RocksDB
- **Stateful deduplication** with watermarks
- **Stream-stream joins** — joining two live streams with bounded state
- **Debugging state** — reading state metrics from `lastProgress`

In [ ]:
import os, time, json, shutil, random
from datetime import datetime, timedelta
from typing import Iterator, Tuple
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, max as _max, min as _min,
    window, session_window, current_timestamp,
    lit, concat, when, floor, expr
)

spark = (
    SparkSession.builder
    .appName("StatefulStreamProcessing")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

BASE   = os.path.dirname(os.path.abspath("17-stateful-stream-processing.ipynb"))
DATA   = os.path.join(BASE, "data")
SCRATCH = os.path.join(BASE, "_scratch_17")
CKPT   = os.path.join(SCRATCH, "ckpt")
SRC    = os.path.join(SCRATCH, "src")
SRC2   = os.path.join(SCRATCH, "src2")

for d in [CKPT, SRC, SRC2]:
    os.makedirs(d, exist_ok=True)

print(f"Spark {spark.version} ready")

In [ ]:
# ── Shared schema and data generator ──────────────────────────────────────────

TXN_SCHEMA = StructType([
    StructField("txn_id",      StringType(),      False),
    StructField("customer_id", StringType(),      False),
    StructField("amount",      DecimalType(18,2), False),
    StructField("category",    StringType(),      True),
    StructField("txn_ts",      TimestampType(),   False),
    StructField("status",      StringType(),      False),
    StructField("is_fraud",    BooleanType(),     False),
])

CATS    = ["FOOD", "TRAVEL", "SHOPPING", "UTILITIES", "ENTERTAINMENT"]
STATUES = ["SUCCESS", "SUCCESS", "SUCCESS", "DECLINED", "PENDING"]

def write_batch(idx: int, src_dir: str = SRC,
                n: int = 40, base_dt: datetime = None,
                customers: list = None):
    """Write n synthetic transaction rows as an ndjson file."""
    if base_dt is None:
        base_dt = datetime(2024, 1, 1, 9, 0, 0) + timedelta(minutes=idx * 5)
    if customers is None:
        customers = [f"CUST{i:04d}" for i in range(1, 11)]
    rows = []
    for i in range(n):
        rows.append({
            "txn_id":      f"T{idx:03d}{i:04d}",
            "customer_id": random.choice(customers),
            "amount":      round(random.uniform(10, 800), 2),
            "category":    random.choice(CATS),
            "txn_ts":      (base_dt + timedelta(seconds=i * 3)).strftime("%Y-%m-%dT%H:%M:%S"),
            "status":      random.choice(STATUES),
            "is_fraud":    random.random() < 0.06,
        })
    path = os.path.join(src_dir, f"batch_{idx:03d}.json")
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

# Pre-populate source directory
for i in range(6):
    write_batch(i)
print("Source files written.")

### Running Aggregations (Unbounded State)

The simplest form of stateful streaming is a `groupBy + agg` **without** a time window. Spark accumulates a running total for every key since the stream started. This state grows indefinitely because there is no window close event to free it.

- Use **complete** output mode — Spark rewrites the full result table every trigger
- No watermark is needed (or useful) — there are no windows to close
- Suitable for small, bounded key spaces (e.g., 5 categories, 50 merchants)
- Avoid for high-cardinality keys (millions of customer IDs) — state will exhaust executor memory

```
Micro-batch 1:  FOOD→3 txns   TRAVEL→2 txns
Micro-batch 2:  FOOD→5 txns   TRAVEL→4 txns   SHOPPING→1 txn
Micro-batch 3:  FOOD→8 txns   TRAVEL→6 txns   SHOPPING→3 txns
                ↑ counts are cumulative — state kept since stream start
```

In [ ]:
src = (
    spark.readStream
    .schema(TXN_SCHEMA)
    .option("maxFilesPerTrigger", 2)
    .json(SRC)
)

running_agg = (
    src
    .filter(col("status") == "SUCCESS")
    .groupBy("category")
    .agg(
        count("txn_id").alias("total_txns"),
        _sum("amount").alias("total_spend"),
        avg("amount").alias("avg_spend"),
    )
)

running_q = (
    running_agg
    .writeStream
    .format("memory").queryName("running_totals")
    .outputMode("complete")   # full result table every trigger
    .option("checkpointLocation", os.path.join(CKPT, "running"))
    .start()
)

# Sample the result mid-stream and after all files are processed
time.sleep(6)
print("=== After first 2 files ===")
spark.sql("SELECT * FROM running_totals ORDER BY total_spend DESC").show()

time.sleep(10)
print("=== After all 6 files ===")
spark.sql("""
    SELECT category,
           total_txns,
           ROUND(total_spend, 2) AS total_spend,
           ROUND(avg_spend, 2)   AS avg_spend
    FROM   running_totals
    ORDER  BY total_spend DESC
""").show()

running_q.stop()

### Session Windows

Tumbling and sliding windows are **fixed-duration** — every window has the same length regardless of the data. A **session window** is **activity-driven**: it opens when the first event for a key arrives and closes after a configurable **gap duration** of inactivity. If a new event arrives before the gap expires, the window extends.

```
customer CUST0001 events:
  09:00 ─── 09:02 ─── 09:04                      09:25 ─── 09:27
  └────────────────────┘  ← gap > 10 min          └──────────┘
     Session 1 (4 min)       → window closes         Session 2

customer CUST0002 events:
  09:01 ─── 09:03 ─── 09:08 ─── 09:12
  └──────────────────────────────────┘ ← one long session
                   Session 1 (11 min)
```

Session windows are ideal for:
- **User session analytics** — how long did a customer browse before transacting?
- **Fraud detection** — bursts of transactions within a short window per card
- **Device heartbeats** — group telemetry events between silence periods

`session_window(event_time_col, gap_duration)` — available since Spark 3.2. The gap duration can be a static string (`"10 minutes"`) or a **dynamic gap** (a column expression returning different gaps per row).

In [ ]:
src2 = (
    spark.readStream
    .schema(TXN_SCHEMA)
    .option("maxFilesPerTrigger", 3)
    .json(SRC)
)

sessions = (
    src2
    .withWatermark("txn_ts", "15 minutes")          # needed to close sessions
    .groupBy(
        session_window(col("txn_ts"), "10 minutes"), # close after 10 min of silence
        col("customer_id")
    )
    .agg(
        count("txn_id").alias("events"),
        _sum("amount").alias("session_spend"),
        _min("txn_ts").alias("session_start"),
        _max("txn_ts").alias("session_end"),
    )
    .select(
        "customer_id",
        col("session_window.start").alias("win_start"),
        col("session_window.end").alias("win_end"),
        "events",
        "session_spend",
    )
)

session_q = (
    sessions
    .writeStream
    .format("memory").queryName("sessions")
    .outputMode("update")
    .option("checkpointLocation", os.path.join(CKPT, "sessions"))
    .start()
)

time.sleep(16)
print("=== Session windows per customer ===")
spark.sql("""
    SELECT customer_id,
           win_start,
           win_end,
           events,
           ROUND(session_spend, 2) AS spend
    FROM   sessions
    ORDER  BY customer_id, win_start
""").show(20, truncate=False)

session_q.stop()

### Dynamic Session Gap

Since Spark 3.2, the gap duration in `session_window` can be a **column expression** — different rows can use different gap sizes. This lets you apply business rules like "high-value customers get a longer session window" without pre-splitting the stream.

In [ ]:
src3 = (
    spark.readStream
    .schema(TXN_SCHEMA)
    .option("maxFilesPerTrigger", 3)
    .json(SRC)
)

# Gap depends on the transaction amount:
#   amount >= 500 → 20-minute gap (high-value, keep the session open longer)
#   amount  < 500 → 5-minute gap
dynamic_gap = (
    when(col("amount") >= 500, lit("20 minutes"))
    .otherwise(lit("5 minutes"))
)

dynamic_sessions = (
    src3
    .withWatermark("txn_ts", "25 minutes")
    .groupBy(
        session_window(col("txn_ts"), dynamic_gap),
        col("customer_id")
    )
    .agg(
        count("txn_id").alias("events"),
        _sum("amount").alias("session_spend"),
    )
    .select(
        "customer_id",
        col("session_window.start").alias("win_start"),
        col("session_window.end").alias("win_end"),
        "events",
        "session_spend",
    )
)

dyn_q = (
    dynamic_sessions
    .writeStream
    .format("memory").queryName("dyn_sessions")
    .outputMode("update")
    .option("checkpointLocation", os.path.join(CKPT, "dyn_sessions"))
    .start()
)

time.sleep(16)
print("=== Dynamic gap sessions ===")
spark.sql("""
    SELECT customer_id, win_start, win_end, events, ROUND(session_spend,2) AS spend
    FROM   dyn_sessions
    ORDER  BY customer_id, win_start
""").show(20, truncate=False)

dyn_q.stop()

### Arbitrary Stateful Processing — mapGroupsWithState

Built-in `groupBy + agg` and `window` cover most use cases. But sometimes you need:
- State that is **not just a counter or sum** (e.g., a list of recent events, a running status machine)
- Custom **expiry logic** beyond watermarks
- Control over **exactly when** to emit output

`mapGroupsWithState` lets you write a Python function that receives all rows for a key in the current micro-batch plus the **previous state** for that key, and returns exactly **one output row** plus the **updated state**.

```
For each key in each micro-batch:
  func(key, new_rows_iterator, old_state) → (output_row, new_state)
```

**Requirements:**
- Input DataFrame must be a Dataset — use `.groupBy().applyInPandas()` in PySpark, or the Scala/Java API for `mapGroupsWithState`
- In PySpark, the equivalent is **`applyInPandasWithState`** (Spark 3.4+) or `mapGroupsWithState` via the Scala API
- For Python, the practical approach is `foreachBatch` with manual state in an external store, or using `applyInPandasWithState`

The section below demonstrates the pattern using `applyInPandasWithState`, the PySpark-native API for arbitrary stateful processing.

In [ ]:
# applyInPandasWithState — PySpark arbitrary stateful processing
# Available in Spark 3.4+
# Tracks per-customer running spend and flags when they cross $1 000 lifetime spend.

import pandas as pd
from pyspark.sql.streaming.state import GroupStateTimeout, GroupState

# Schema for each output row
OUTPUT_SCHEMA = StructType([
    StructField("customer_id",    StringType(),      False),
    StructField("batch_spend",    DoubleType(),      False),
    StructField("lifetime_spend", DoubleType(),      False),
    StructField("txn_count",      LongType(),        False),
    StructField("threshold_hit",  BooleanType(),     False),
    StructField("updated_at",     TimestampType(),   False),
])

# Schema for the state stored per key
STATE_SCHEMA = StructType([
    StructField("lifetime_spend", DoubleType(), False),
    StructField("txn_count",      LongType(),   False),
    StructField("threshold_hit",  BooleanType(), False),
])

def update_customer_state(
    key: Tuple,
    rows: Iterator[pd.DataFrame],
    state: GroupState,
) -> Iterator[pd.DataFrame]:
    """
    Called once per customer_id per micro-batch.
    key   : (customer_id,)
    rows  : iterator of pandas DataFrames for this key in this batch
    state : previous state + mutation API
    """
    customer_id = key[0]

    # Load previous state or initialise
    if state.exists:
        prev = state.get
        lifetime_spend = float(prev["lifetime_spend"][0])
        txn_count      = int(prev["txn_count"][0])
        threshold_hit  = bool(prev["threshold_hit"][0])
    else:
        lifetime_spend = 0.0
        txn_count      = 0
        threshold_hit  = False

    # Handle timeout — emit a summary row and remove state
    if state.hasTimedOut:
        state.remove()
        yield pd.DataFrame([{
            "customer_id":    customer_id,
            "batch_spend":    0.0,
            "lifetime_spend": lifetime_spend,
            "txn_count":      txn_count,
            "threshold_hit":  threshold_hit,
            "updated_at":     pd.Timestamp.now(),
        }])
        return

    # Accumulate rows from this micro-batch
    batch_df    = pd.concat(list(rows))
    success     = batch_df[batch_df["status"] == "SUCCESS"]
    batch_spend = float(success["amount"].sum()) if len(success) > 0 else 0.0
    batch_count = len(success)

    lifetime_spend += batch_spend
    txn_count      += batch_count
    threshold_hit   = threshold_hit or (lifetime_spend >= 1000.0)

    # Save updated state
    state.update(pd.DataFrame([{
        "lifetime_spend": lifetime_spend,
        "txn_count":      txn_count,
        "threshold_hit":  threshold_hit,
    }]))

    # Set a processing-time timeout: remove state if no events for 60 seconds
    state.setTimeoutDuration(60_000)  # ms

    # Emit exactly one summary row for this customer this batch
    yield pd.DataFrame([{
        "customer_id":    customer_id,
        "batch_spend":    batch_spend,
        "lifetime_spend": lifetime_spend,
        "txn_count":      txn_count,
        "threshold_hit":  threshold_hit,
        "updated_at":     pd.Timestamp.now(),
    }])

print("State update function defined.")

In [ ]:
src4 = (
    spark.readStream
    .schema(TXN_SCHEMA)
    .option("maxFilesPerTrigger", 2)
    .json(SRC)
)

customer_state_stream = (
    src4
    .groupBy("customer_id")
    .applyInPandasWithState(
        update_customer_state,
        outputStructType  = OUTPUT_SCHEMA,
        stateStructType   = STATE_SCHEMA,
        outputMode        = "update",
        timeoutConf       = GroupStateTimeout.ProcessingTimeTimeout,
    )
)

state_q = (
    customer_state_stream
    .writeStream
    .format("memory").queryName("customer_state")
    .outputMode("update")
    .option("checkpointLocation", os.path.join(CKPT, "customer_state"))
    .start()
)

time.sleep(16)

print("=== Per-customer running state ===")
spark.sql("""
    SELECT customer_id,
           txn_count,
           ROUND(lifetime_spend, 2) AS lifetime_spend,
           threshold_hit
    FROM   customer_state
    ORDER  BY lifetime_spend DESC
""").show()

print("=== Customers who crossed $1 000 lifetime spend ===")
spark.sql("""
    SELECT customer_id, txn_count, ROUND(lifetime_spend,2) AS spend
    FROM   customer_state
    WHERE  threshold_hit = true
    ORDER  BY spend DESC
""").show()

state_q.stop()

### Timeouts — Expiring Idle State

Without timeouts, state for inactive keys stays in the state store forever — consuming memory. Timeouts let Spark call your state function one final time for a key that has not received events for a configurable duration, giving you the chance to emit a final result and remove the state.

| Timeout type | API | Clock used | Typical use |
|---|---|---|---|
| **ProcessingTimeTimeout** | `state.setTimeoutDuration(ms)` | Wall-clock time | Remove idle state after N minutes of real time |
| **EventTimeTimeout** | `state.setTimeoutTimestamp(epoch_ms)` | Watermark | Remove state once watermark passes a threshold event time |
| **NoTimeout** | `GroupStateTimeout.NoTimeout` | — | State lives until explicitly removed |

**When a timeout fires:**
- Spark calls your state function with an empty row iterator and `state.hasTimedOut == True`
- You can emit a final row (e.g., a session-closed event) and call `state.remove()`
- If you do not call `state.remove()`, the timeout keeps firing

**ProcessingTimeTimeout** — simpler; fires N milliseconds after the last time you called `setTimeoutDuration`. Not affected by event time or watermarks.

**EventTimeTimeout** — fires when the watermark advances past the timestamp you set with `setTimeoutTimestamp`. Deterministic and reproducible across replays.

In [ ]:
# Fraud burst detector using EventTimeTimeout
# Flags customers who make ≥3 transactions within a 3-minute window.
# State expires 10 minutes after the last event.

BURST_SCHEMA = StructType([
    StructField("customer_id",  StringType(),    False),
    StructField("burst_count",  LongType(),      False),
    StructField("burst_amount", DoubleType(),    False),
    StructField("first_ts",     TimestampType(), True),
    StructField("last_ts",      TimestampType(), True),
    StructField("alert",        BooleanType(),   False),
    StructField("expired",      BooleanType(),   False),
])

BURST_STATE_SCHEMA = StructType([
    StructField("count",       LongType(),   False),
    StructField("total",       DoubleType(), False),
    StructField("first_ts_ms", LongType(),   False),
    StructField("last_ts_ms",  LongType(),   False),
])

BURST_THRESHOLD = 3
BURST_WINDOW_SEC = 180    # 3 minutes
TIMEOUT_SEC      = 600    # 10 minutes event-time timeout

def detect_burst(
    key: Tuple,
    rows: Iterator[pd.DataFrame],
    state: GroupState,
) -> Iterator[pd.DataFrame]:
    customer_id = key[0]

    if state.hasTimedOut:
        # Emit expiry record and clean up
        prev = state.get if state.exists else None
        state.remove()
        if prev is not None:
            yield pd.DataFrame([{
                "customer_id":  customer_id,
                "burst_count":  int(prev["count"][0]),
                "burst_amount": float(prev["total"][0]),
                "first_ts":     pd.Timestamp(int(prev["first_ts_ms"][0]), unit="ms"),
                "last_ts":      pd.Timestamp(int(prev["last_ts_ms"][0]),  unit="ms"),
                "alert":        False,
                "expired":      True,
            }])
        return

    batch = pd.concat(list(rows))
    success = batch[batch["status"] == "SUCCESS"]
    if len(success) == 0:
        return

    # Convert timestamps
    success = success.copy()
    success["ts_ms"] = pd.to_datetime(success["txn_ts"]).astype("int64") // 1_000_000

    if state.exists:
        prev        = state.get
        count       = int(prev["count"][0])
        total       = float(prev["total"][0])
        first_ts_ms = int(prev["first_ts_ms"][0])
        last_ts_ms  = int(prev["last_ts_ms"][0])
    else:
        count = 0; total = 0.0
        first_ts_ms = int(success["ts_ms"].min())
        last_ts_ms  = first_ts_ms

    for _, row in success.iterrows():
        ts_ms = int(row["ts_ms"])
        # Reset burst window if gap exceeds BURST_WINDOW_SEC
        if ts_ms - last_ts_ms > BURST_WINDOW_SEC * 1000:
            count = 0; total = 0.0; first_ts_ms = ts_ms
        count      += 1
        total      += float(row["amount"])
        last_ts_ms  = ts_ms

    alert = count >= BURST_THRESHOLD

    state.update(pd.DataFrame([{
        "count":       count,
        "total":       total,
        "first_ts_ms": first_ts_ms,
        "last_ts_ms":  last_ts_ms,
    }]))
    # EventTimeTimeout: expire state 10 min after last event
    state.setTimeoutTimestamp(last_ts_ms + TIMEOUT_SEC * 1000)

    yield pd.DataFrame([{
        "customer_id":  customer_id,
        "burst_count":  count,
        "burst_amount": round(total, 2),
        "first_ts":     pd.Timestamp(first_ts_ms, unit="ms"),
        "last_ts":      pd.Timestamp(last_ts_ms,  unit="ms"),
        "alert":        alert,
        "expired":      False,
    }])

print("Burst detector defined.")

In [ ]:
src5 = (
    spark.readStream
    .schema(TXN_SCHEMA)
    .option("maxFilesPerTrigger", 2)
    .json(SRC)
)

burst_stream = (
    src5
    .withWatermark("txn_ts", "10 minutes")
    .groupBy("customer_id")
    .applyInPandasWithState(
        detect_burst,
        outputStructType = BURST_SCHEMA,
        stateStructType  = BURST_STATE_SCHEMA,
        outputMode       = "append",
        timeoutConf      = GroupStateTimeout.EventTimeTimeout,
    )
)

burst_q = (
    burst_stream
    .writeStream
    .format("memory").queryName("burst_alerts")
    .outputMode("append")
    .option("checkpointLocation", os.path.join(CKPT, "burst"))
    .start()
)

time.sleep(18)

print("=== Burst alerts (≥3 transactions within 3 minutes) ===")
spark.sql("""
    SELECT customer_id, burst_count, ROUND(burst_amount,2) AS amount,
           first_ts, last_ts, alert, expired
    FROM   burst_alerts
    WHERE  alert = true
    ORDER  BY last_ts
""").show(truncate=False)

burst_q.stop()

### flatMapGroupsWithState — Emit Multiple Rows Per Key

`mapGroupsWithState` emits exactly one row per key per batch. `flatMapGroupsWithState` (the PySpark equivalent is `applyInPandasWithState` with `outputMode="append"`) can emit **zero or many rows** per key per batch.

This is useful when:
- You want to emit one row **per transaction** that changes the state (not just one summary row)
- You want to emit nothing for some batches and a burst of rows when a threshold is hit
- You are implementing a state machine where transitions emit events

The fraud burst detector above already uses `append` output mode (zero or one row), but the function can `yield` any number of `pd.DataFrame` chunks.

In [ ]:
# State machine: track each customer's spending tier
# BRONZE → SILVER (>=$500 lifetime) → GOLD (>=$2000 lifetime)
# Emit a tier-change event row ONLY when the tier changes.

TIER_EVENT_SCHEMA = StructType([
    StructField("customer_id",    StringType(),    False),
    StructField("old_tier",       StringType(),    True),
    StructField("new_tier",       StringType(),    False),
    StructField("lifetime_spend", DoubleType(),    False),
    StructField("changed_at",     TimestampType(), False),
])

TIER_STATE_SCHEMA = StructType([
    StructField("lifetime_spend", DoubleType(), False),
    StructField("tier",           StringType(), False),
])

def get_tier(spend: float) -> str:
    if spend >= 2000:
        return "GOLD"
    elif spend >= 500:
        return "SILVER"
    return "BRONZE"

def track_tier_changes(
    key: Tuple,
    rows: Iterator[pd.DataFrame],
    state: GroupState,
) -> Iterator[pd.DataFrame]:
    customer_id = key[0]

    if state.hasTimedOut:
        state.remove()
        return   # emit nothing on timeout

    prev_spend = 0.0
    prev_tier  = "BRONZE"
    if state.exists:
        s = state.get
        prev_spend = float(s["lifetime_spend"][0])
        prev_tier  = str(s["tier"][0])

    batch = pd.concat(list(rows))
    success = batch[batch["status"] == "SUCCESS"]

    new_spend = prev_spend + (float(success["amount"].sum()) if len(success) > 0 else 0.0)
    new_tier  = get_tier(new_spend)

    state.update(pd.DataFrame([{
        "lifetime_spend": new_spend,
        "tier":           new_tier,
    }]))
    state.setTimeoutDuration(120_000)  # expire after 2 min of processing-time inactivity

    # Emit a row ONLY when the tier changed — zero rows otherwise
    if new_tier != prev_tier:
        yield pd.DataFrame([{
            "customer_id":    customer_id,
            "old_tier":       prev_tier,
            "new_tier":       new_tier,
            "lifetime_spend": round(new_spend, 2),
            "changed_at":     pd.Timestamp.now(),
        }])
    # else: yield nothing — flatMapGroupsWithState emits zero rows


src6 = (
    spark.readStream
    .schema(TXN_SCHEMA)
    .option("maxFilesPerTrigger", 2)
    .json(SRC)
)

tier_stream = (
    src6
    .groupBy("customer_id")
    .applyInPandasWithState(
        track_tier_changes,
        outputStructType = TIER_EVENT_SCHEMA,
        stateStructType  = TIER_STATE_SCHEMA,
        outputMode       = "append",
        timeoutConf      = GroupStateTimeout.ProcessingTimeTimeout,
    )
)

tier_q = (
    tier_stream
    .writeStream
    .format("memory").queryName("tier_changes")
    .outputMode("append")
    .option("checkpointLocation", os.path.join(CKPT, "tier"))
    .start()
)

time.sleep(18)

print("=== Tier upgrade events ===")
spark.sql("""
    SELECT customer_id, old_tier, new_tier, lifetime_spend, changed_at
    FROM   tier_changes
    ORDER  BY changed_at
""").show(truncate=False)

tier_q.stop()

### Stream-Stream Join

Joining two live streams requires both sides to buffer rows in state until a match is found. Without bounds, the buffers grow indefinitely. **Watermarks on both sides** tell Spark when it is safe to evict unmatched rows.

```
Stream A: card transactions   (watermark: txn_ts, 10 min)
Stream B: fraud alerts        (watermark: alert_ts, 10 min)

Inner join condition:
  A.txn_id = B.txn_id
  AND A.txn_ts BETWEEN B.alert_ts - 5 min AND B.alert_ts + 5 min
                ↑
  Time range required — lets Spark evict rows from A whose window
  can never match any remaining row in B.
```

**State eviction rule for inner join:**
- A row from stream A at time `t` can only match rows from stream B in the range `[t - range, t + range]`
- Once the watermark on stream B advances past `t + range`, no new B rows can match this A row — it is evicted

**Supported join types:**

| Type | Watermark required | Notes |
|---|---|---|
| inner | Yes (both sides) | Time range condition required for state cleanup |
| left outer | Yes (both sides) | Right-side unmatched rows emitted after watermark advances |
| right outer | Yes (both sides) | Left-side unmatched rows emitted after watermark advances |
| full outer | Not supported | — |

In [ ]:
# Create a second stream: fraud confirmation signals arriving a few minutes after transactions
import random

FRAUD_SCHEMA = StructType([
    StructField("txn_id",    StringType(),    False),
    StructField("alert_ts",  TimestampType(), False),
    StructField("rule",      StringType(),    False),
    StructField("score",     DoubleType(),    False),
])

# Read back the transactions already written to SRC to generate matching alert files
existing_txns = spark.read.schema(TXN_SCHEMA).json(SRC)
fraud_txn_ids = (
    existing_txns
    .filter(col("is_fraud") == True)
    .select("txn_id", "txn_ts")
    .limit(20)
    .collect()
)

# Write alert files to SRC2 — each alert arrives 1-4 minutes after the transaction
rules = ["VELOCITY_CHECK", "GEOLOCATION", "AMOUNT_ANOMALY", "DEVICE_MISMATCH"]
for i, row in enumerate(fraud_txn_ids):
    alert_ts = row["txn_ts"] + timedelta(minutes=random.randint(1, 4))
    alert = {
        "txn_id":   row["txn_id"],
        "alert_ts": alert_ts.strftime("%Y-%m-%dT%H:%M:%S"),
        "rule":     random.choice(rules),
        "score":    round(random.uniform(0.6, 0.99), 3),
    }
    path = os.path.join(SRC2, f"alert_{i:04d}.json")
    with open(path, "w") as f:
        f.write(json.dumps(alert) + "\n")

print(f"Written {len(fraud_txn_ids)} alert files to {SRC2}")

In [ ]:
txn_stream = (
    spark.readStream
    .schema(TXN_SCHEMA)
    .option("maxFilesPerTrigger", 2)
    .json(SRC)
    .withWatermark("txn_ts", "10 minutes")
)

alert_stream = (
    spark.readStream
    .schema(FRAUD_SCHEMA)
    .option("maxFilesPerTrigger", 10)
    .json(SRC2)
    .withWatermark("alert_ts", "10 minutes")
)

# Stream-stream inner join with time-range constraint
confirmed_fraud = (
    txn_stream.alias("t")
    .join(
        alert_stream.alias("a"),
        expr("""
            t.txn_id = a.txn_id
            AND a.alert_ts >= t.txn_ts
            AND a.alert_ts <= t.txn_ts + INTERVAL 5 MINUTES
        """),
        how="inner",
    )
    .select(
        col("t.txn_id"),
        col("t.customer_id"),
        col("t.amount"),
        col("t.txn_ts"),
        col("a.alert_ts"),
        col("a.rule"),
        col("a.score"),
    )
)

join_q = (
    confirmed_fraud
    .writeStream
    .format("memory").queryName("confirmed_fraud")
    .outputMode("append")
    .option("checkpointLocation", os.path.join(CKPT, "stream_join"))
    .start()
)

time.sleep(20)

print("=== Confirmed fraud: transaction ⋈ alert (stream-stream join) ===")
spark.sql("""
    SELECT txn_id, customer_id, ROUND(amount,2) AS amount,
           txn_ts, alert_ts, rule, score
    FROM   confirmed_fraud
    ORDER  BY txn_ts
""").show(20, truncate=False)

join_q.stop()

### Stateful Deduplication

Sources like Kafka guarantee **at-least-once** delivery — the same message can appear more than once (on broker failover, consumer rebalance, or retry). `dropDuplicates` on a unique key removes duplicates within the state window.

Without a watermark, `dropDuplicates` holds every seen key forever. With a watermark, keys older than the watermark are evicted — any duplicate arriving after that window is treated as a new event (acceptable trade-off for bounded memory).

```
.withWatermark("txn_ts", "10 minutes")
.dropDuplicates(["txn_id", "txn_ts"])
```

The deduplication key should include the event-time column so state eviction aligns with the watermark.

In [ ]:
# Create a source with deliberate duplicates
DEDUP_SRC = os.path.join(SCRATCH, "dedup_src")
os.makedirs(DEDUP_SRC, exist_ok=True)

base_dt = datetime(2024, 1, 1, 9, 0, 0)
rows = [
    {"txn_id": "TX001", "customer_id": "CUST0001", "amount": 120.0,
     "category": "FOOD",     "txn_ts": "2024-01-01T09:01:00", "status": "SUCCESS", "is_fraud": False},
    {"txn_id": "TX002", "customer_id": "CUST0002", "amount": 450.0,
     "category": "TRAVEL",   "txn_ts": "2024-01-01T09:02:00", "status": "SUCCESS", "is_fraud": False},
    {"txn_id": "TX001", "customer_id": "CUST0001", "amount": 120.0,  # DUPLICATE of TX001
     "category": "FOOD",     "txn_ts": "2024-01-01T09:01:00", "status": "SUCCESS", "is_fraud": False},
    {"txn_id": "TX003", "customer_id": "CUST0003", "amount":  80.0,
     "category": "SHOPPING", "txn_ts": "2024-01-01T09:03:00", "status": "SUCCESS", "is_fraud": False},
    {"txn_id": "TX002", "customer_id": "CUST0002", "amount": 450.0,  # DUPLICATE of TX002
     "category": "TRAVEL",   "txn_ts": "2024-01-01T09:02:00", "status": "SUCCESS", "is_fraud": False},
]
with open(os.path.join(DEDUP_SRC, "dupes.json"), "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")

print(f"Written {len(rows)} rows (including 2 duplicates)")

dedup_src = (
    spark.readStream
    .schema(TXN_SCHEMA)
    .json(DEDUP_SRC)
)

deduped = (
    dedup_src
    .withWatermark("txn_ts", "10 minutes")
    .dropDuplicates(["txn_id", "txn_ts"])   # deduplicate on business key + event time
)

dedup_q = (
    deduped
    .writeStream
    .format("memory").queryName("deduped")
    .outputMode("append")
    .option("checkpointLocation", os.path.join(CKPT, "dedup"))
    .trigger(once=True)
    .start()
)
dedup_q.awaitTermination()

raw_count   = spark.read.schema(TXN_SCHEMA).json(DEDUP_SRC).count()
dedup_count = spark.sql("SELECT count(*) FROM deduped").collect()[0][0]

print(f"Raw rows    : {raw_count}    (includes duplicates)")
print(f"After dedup : {dedup_count}  (unique txn_id + txn_ts pairs)")
spark.sql("SELECT txn_id, customer_id, amount, txn_ts FROM deduped ORDER BY txn_ts").show()

### State Store Backends

Spark stores stateful aggregation state in a **state store**. Two backends are available:

| Backend | Config value | Characteristics |
|---|---|---|
| **HDFS-backed** (default) | `HDFSBackedStateStoreProvider` | State is stored in memory and checkpointed to HDFS/S3 after each batch. Simple, no extra dependencies. Can struggle with very large state (GBs per executor). |
| **RocksDB** | `RocksDBStateStoreProvider` | State is stored on-disk (RocksDB, a local key-value store) and checkpointed to HDFS/S3. Handles much larger state with lower heap pressure. Requires Spark 3.2+ and the RocksDB native library. |

**When to use RocksDB:**
- State size per executor exceeds a few GB
- You see frequent GC pauses or OOM errors on stateful stages
- High-cardinality keys (millions of customer IDs, device IDs)

```python
spark = SparkSession.builder
    .config(
        "spark.sql.streaming.stateStore.providerClass",
        "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider"
    )
    .getOrCreate()
```

In [ ]:
# Inspect current state store configuration
state_store_key = "spark.sql.streaming.stateStore.providerClass"
try:
    provider = spark.conf.get(state_store_key)
except Exception:
    provider = "(default — HDFSBackedStateStoreProvider)"

print(f"State store provider : {provider}")
print()
print("To switch to RocksDB, set before creating the SparkSession:")
print("  spark.sql.streaming.stateStore.providerClass")
print("    = org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider")

### Monitoring State — stateOperators in lastProgress

Every stateful streaming query reports state metrics in `lastProgress["stateOperators"]`. This is the primary tool for diagnosing state growth, memory pressure, and timeout behaviour.

Key fields per operator:

| Field | Meaning |
|---|---|
| `numRowsTotal` | Total rows currently in the state store |
| `numRowsUpdated` | Rows added or updated in this micro-batch |
| `numRowsRemoved` | Rows evicted (timeout or watermark) in this micro-batch |
| `memoryUsedBytes` | State store memory in bytes |
| `numShufflePartitions` | Number of state partitions (= `spark.sql.shuffle.partitions`) |
| `operatorName` | Name of the stateful operator (e.g., `stateStoreSave`, `symmetricHashJoin`) |

In [ ]:
import pprint

monitor_src = (
    spark.readStream
    .schema(TXN_SCHEMA)
    .option("maxFilesPerTrigger", 1)
    .json(SRC)
)

monitor_agg = (
    monitor_src
    .withWatermark("txn_ts", "10 minutes")
    .groupBy(window(col("txn_ts"), "5 minutes"), col("category"))
    .agg(count("txn_id").alias("n"))
)

monitor_q = (
    monitor_agg
    .writeStream
    .format("memory").queryName("monitor_state")
    .outputMode("update")
    .option("checkpointLocation", os.path.join(CKPT, "monitor"))
    .start()
)

time.sleep(14)

p = monitor_q.lastProgress
if p:
    print(f"Batch id          : {p['batchId']}")
    print(f"Input rows        : {p['numInputRows']}")
    print(f"Watermark         : {p.get('eventTime', {}).get('watermark', 'n/a')}")
    print()
    for op in p.get("stateOperators", []):
        print("State operator:")
        print(f"  operatorName     : {op.get('operatorName', 'n/a')}")
        print(f"  numRowsTotal     : {op.get('numRowsTotal', 0)}")
        print(f"  numRowsUpdated   : {op.get('numRowsUpdated', 0)}")
        print(f"  numRowsRemoved   : {op.get('numRowsRemoved', 0)}")
        print(f"  memoryUsedBytes  : {op.get('memoryUsedBytes', 0):,}")

monitor_q.stop()

### Choosing the Right Stateful Pattern

Use this decision tree to pick the right API:

```
Do you need state across micro-batches?
│
├── No  → stateless: filter, select, stream-static join — use append mode
│
└── Yes
    │
    ├── Fixed-duration windows? (every 5 min, every hour)
    │   └── groupBy + window() + withWatermark  ← tumbling/sliding windows
    │
    ├── Activity-driven windows? (close on silence)
    │   └── groupBy + session_window() + withWatermark  ← session windows
    │
    ├── Running totals / counts (small key space)?
    │   └── groupBy + agg, outputMode=complete  ← unbounded running agg
    │
    ├── Per-key custom state (state machine, complex accumulation)?
    │   ├── Emit one row per key per batch → applyInPandasWithState (update)
    │   └── Emit 0..N rows per key per batch → applyInPandasWithState (append)
    │
    ├── Deduplication?
    │   └── withWatermark + dropDuplicates([key, event_time_col])
    │
    └── Join two live streams?
        └── stream-stream join with watermarks on both sides + time range condition
```

**Memory cost guide:**

| Pattern | State size | Memory risk |
|---|---|---|
| Tumbling/sliding window + watermark | Bounded by watermark × key space | Low |
| Session window + watermark | Bounded by gap × active sessions | Low–Medium |
| Running agg (no window) | Grows with unique keys | High for large key spaces |
| `applyInPandasWithState` + timeout | Bounded by timeout + key space | Medium |
| Dedup + watermark | Bounded by watermark × unique keys | Medium |
| Stream-stream join + watermarks | Bounded by watermark × join rate | Medium–High |

### Summary

**Stateful streaming** maintains per-key state across micro-batches. Spark stores this state in the **state store** (HDFS-backed by default; RocksDB for large state).

**Running aggregations** (`groupBy + agg` without a window) accumulate state forever. Use `complete` output mode; limit to small, bounded key spaces.

**Session windows** (`session_window(col, gap)`) are activity-driven — they close after a configurable inactivity gap. The gap can be static or a dynamic column expression. Require a watermark for state cleanup.

**`applyInPandasWithState`** (PySpark 3.4+) is the arbitrary stateful API:
- You receive a key, an iterator of pandas DataFrames for the current batch, and the previous state
- You return zero or more output DataFrames and update the state
- Use for state machines, complex accumulations, or anything `groupBy + agg` cannot express

**Timeouts** expire idle state:
- `ProcessingTimeTimeout` — fires after N milliseconds of wall-clock inactivity
- `EventTimeTimeout` — fires when the watermark advances past a threshold event time
- When fired, your function receives `state.hasTimedOut == True`; call `state.remove()` to free state

**Stream-stream joins** buffer both sides in state waiting for matches. Both sides need watermarks and a time-range join condition to enable state eviction.

**Stateful deduplication** — `withWatermark + dropDuplicates([key, event_time_col])` — removes duplicates within the watermark window with bounded state.

**Monitoring** — `lastProgress["stateOperators"]` shows `numRowsTotal`, `numRowsUpdated`, `numRowsRemoved`, and `memoryUsedBytes` for every stateful operator in the query.

In [ ]:
# Cleanup
if os.path.exists(SCRATCH):
    shutil.rmtree(SCRATCH)
    print(f"Removed: {SCRATCH}")
print("Cleanup complete.")